Turn the Function Into a LangChain Tool

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from typing import Optional
import math
import traceback

Our First Tool

In [3]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    
    try:
        return str(eval(expression))

    except Exception as e:
        raise Exception(
            f"Calculator failed: {e}"
        )

Second Tool: Search

In [ ]:
@tool
def search(query: str) -> str:
    """
    Search for information about a topic.

    Use this tool when the user explicitly asks
    to search for information.
    """

    # Simulate a tool failure
    if "404 test" in query.lower():
        raise Exception(
            "Search service unavailable."
        )

    # Simulated search results
    results = {
        "tunisia": (
            "Tunisia is a country in North Africa. "
            "Its capital is Tunis. Tunisia is located "
            "on the Mediterranean coast and is known "
            "for its history, culture, and tourism."
        ),

        "python": (
            "Python is a high-level programming language "
            "known for its simplicity and wide use in "
            "web development, data science, automation, "
            "and artificial intelligence."
        )
    }

    query_lower = query.lower()

    for keyword, result in results.items():

        if keyword in query_lower:
            return result

    return (
        f"No detailed search results were found for: "
        f"{query}"
    )

Test

The Agent

In [23]:
from langchain_ollama import ChatOllama
# model
model = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [24]:
tools = [
    calculator,
    search
]

tool_map = {
    "calculator": calculator,
    "search": search
}

print(tool_map.keys())



dict_keys(['calculator', 'search'])


In [26]:
model_with_tools = model.bind_tools(tools)

tool executor

In [36]:
def run_agent(user_input: str) -> dict:
    """
    Simple agent loop with explicit fallback.
    Returns a dict containing the answer and the route taken.
    """
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant. "
            "Use the available tools when they can help answer the question accurately. "
            "If a tool is not needed or fails, answer directly from your knowledge."
        )),
        HumanMessage(content=user_input)
    ]
    
    route = "direct_llm"          # default
    final_answer = None
    
    try:
        # First call – model may decide to call tools
        response = model_with_tools.invoke(messages)
        messages.append(response)
        
        # If the model requested tools
        if response.tool_calls:
            route = "tool"
            for tool_call in response.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                print(f"→ Calling tool: {tool_name} with {tool_args}")
                
                try:
                    tool_result = tool_map[tool_name].invoke(tool_args)
                    print(f"  Tool result: {tool_result}")
                    
                    # Feed the tool result back
                    messages.append(
                        ToolMessage(
                            content=str(tool_result),
                            tool_call_id=tool_call["id"]
                        )
                    )
                except Exception as e:
                    # Tool failed → switch to fallback
                    print(f"  Tool FAILED: {e}")
                    route = "fallback"
                    # Tell the model the tool failed
                    messages.append(
                        ToolMessage(
                            content=f"Tool error: {str(e)}. Please answer without the tool.",
                            tool_call_id=tool_call["id"]
                        )
                    )
            
            # Second call – model produces final answer after seeing tool results
            final_response = model_with_tools.invoke(messages)
            final_answer = final_response.content
        else:
            # Model answered directly (no tool needed)
            final_answer = response.content
            route = "direct_llm"
            
    except Exception as e:
        # Any unexpected error → pure fallback
        print(f"Agent error, falling back to pure LLM: {e}")
        route = "fallback"
        pure_response = model_with_tools.invoke([HumanMessage(content=user_input)])
        final_answer = pure_response.content
    
    return {
        "answer": final_answer,
        "route": route          # "tool" | "direct_llm" | "fallback"
    }

Test

In [37]:
def ask(query: str):
    print(f"\n{'='*60}")
    print(f"USER: {query}")
    result = run_agent(query)
    print(f"ROUTE: {result['route'].upper()}")
    print(f"ANSWER: {result['answer']}")
    print(f"{'='*60}")
    return result

In [38]:
# 1. Query that should use the Calculator tool
ask("What is sqrt(144) + 5?")

# 2. Query that should use Calculator
ask("What is 12 * 8?")

# 3. Simulate tool failure → must fallback
ask("Search 404 test")

# 4. Query that can be answered directly (no tool needed)
ask("Who are you?")

# 5. Query that may use Search or answer directly
ask("Tell me something about Tunisia")


USER: What is sqrt(144) + 5?
→ Calling tool: calculator with {'expression': 'sqrt(144) + 5'}
  Tool FAILED: Calculator failed: name 'sqrt' is not defined
ROUTE: FALLBACK
ANSWER: To calculate the square root of 144, we can use the fact that 12^2 = 144. So, sqrt(144) = 12.

Now, let's add 5 to 12: 12 + 5 = 17.

USER: What is 12 * 8?
→ Calling tool: calculator with {'expression': '12 * 8'}
  Tool result: 96
ROUTE: TOOL
ANSWER: The answer to 12 * 8 is 96.

USER: Search 404 test
→ Calling tool: search with {'query': '404 test'}
  Tool FAILED: Search service unavailable.
ROUTE: FALLBACK
ANSWER: A 404 test is a test to verify that a web server is returning a 404 Not Found error for a non-existent URL. This test is often used to ensure that a web server is functioning correctly and returning the expected error code for requests that cannot be fulfilled.

To perform a 404 test, you can try accessing a URL that does not exist, such as `http://example.com/non-existent-page`. If the server is ret

{'answer': 'The country has a rich history, with ancient civilizations such as Carthage and Rome having left their mark. The country is also known for its beautiful beaches, mountains, and deserts. Tunisia has a diverse culture, with a mix of Arab, Berber, and French influences. The official language is Arabic, but French and Berber are also widely spoken.\n\nSome popular tourist destinations in Tunisia include:\n\n* The ancient city of Carthage, a UNESCO World Heritage Site\n* The medina (old city) of Tunis, a UNESCO World Heritage Site\n* The beaches of Hammamet and Sousse\n* The mountains of Djerba and the Sahara Desert\n* The ancient city of Dougga, a UNESCO World Heritage Site\n\nTunisia has a diverse economy, with a mix of agriculture, industry, and services. The country is also known for its textiles, leather goods, and tourism industry.\n\nThe country has a population of around 12 million people, with a mix of Arab, Berber, and French influences. The official language is Arabic